In this notebook feature engineering and data cleaning are performed. The workflow is:
1. removing some features 
3. handling NaN values
2. adding feature to distinguish the single-hit event, *less than 2-hit* from multi-hit event



# 0 - General Ops

In [1]:
import pandas as pd
import numpy as np

import os
from pathlib import Path


In [2]:
# To properly visualization
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# To reset the options to their default values uncomment the following lines and run the cell
# pd.reset_option("display.max_rows")
# pd.reset_option("display.max_columns")
# pd.reset_option("display.max_colwidth")
# pd.reset_option("display.width")

In [ ]:
# ## go 2 dir up to the project root folder
# project_dir = Path(os.getcwd()).parent.parent
# os.chdir(project_dir)
# file_path = project_dir / "data" / "Background" / "Engineered_Datasets" 
# file_name = "combinedbg.csv"

file_path = "../../data/Background/Engineered_Datasets"
file_name = "/combinedbg_full.csv"
SAVE = False

In [4]:
# ds = pd.read_csv(file_path / file_name, header=0)
ds = pd.read_csv(file_path + file_name, header=0)
ds.shape

(368775, 54)

In [5]:
ds.describe(include="all")

,event_index,pixel_ijk,primary_e0_kev,n_voxels,E_sum,E_mean,E_std,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,E_skew,E_kurt,entropy,gini,n_active,frac_active,frac_top1,frac_top3,eff_E_ge_T,com_x,com_y,com_z,rg,lam1,lam2,lam3,linearity,planarity,sphericity,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,hits_mean_active,corr_E_hits,D_f,D_f_r2,f_occ,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,368775.000000,368775,3.687750e+05,368775.000000,3.687750e+05,368775.000000,126750.000000,368775.000000,3.687750e+05,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,3.687750e+05,368775.000000,368775.000000,84747.000000,84747.000000,368775.000000,368775.000000,368775.000000,368775.0,368775.000000,368775.000000,368775.0,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,3.687750e+05,368775.000000,126750.000000,1.267500e+05,1.267500e+05,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,368775.000000,368775.0,0.0,9.748100e+04,78077.000000,0.0,368775.000000,368775.000000,368775.000000,368775.000000,368775
unique,NaN,99839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,NaN,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,NaN,1827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152860
mean,88665.919593,NaN,3.788122e+06,5.183087,4.829744e+04,2772.995719,9072.935762,1075.876745,1.551659e+04,1916.510729,1199.782782,1408.801879,2972.848833,4622.593327,6474.335505,1.198532e+04,1564.046954,615.224399,1.297904,3.397138,0.396060,0.030311,5.183087,1.0,0.846003,0.955666,1.0,4.485840,4.454924,4.495729,0.532421,1.036786,8.060258e-02,0.025634,0.907293,7.006472e-02,2.264238e-02,-0.000523,-0.000245,-0.000254,1.106927,1.109506,1.096747,5.183087,1.0,NaN,6.652459e-01,0.943785,NaN,0.879382,0.968677,2.555225,5.964878,NaN
std,83229.161397,NaN,2.732136e+07,16.569414,2.553609e+05,5986.373762,15664.348947,4121.637277,6.948803e+04,5162.662125,4207.609998,4448.432230,6690.789365,10062.714948,16013.212976,4.330808e+04,4292.909057,2014.594375,1.737200,12.103889,0.691132,0.055052,16.569414,0.0,0.251401,0.120661,0.0,3.142488,3.083255,3.126597,0.927120,2.487684,2.866767e-01,0.105961,0.145486,1.225022e-01,4.690550e-02,0.272683,0.273376,0.280780,2.303389,2.295562,2.279172,16.569414,0.0,NaN,4.557233e-01,0.079299,NaN,2.148318,2.226123,13.115304,20.636104,NaN
min,1.000000,NaN,1.507898e+01,1.000000,1.500171e+01,15.001710,0.003062,15.000290,1.500171e+01,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,1.500171e+01,0.000000,0.000000,-1.967835,-2.437490,-0.000000,0.000000,1.000000,1.0,0.052516,0.140061,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.007281,0.000000e+00,0.000000e+00,-0.999577,-0.999824,-0.999575,0.000000,0.000000,0.000000,1.000000,1.0,NaN,-1.057477e-16,0.600000,NaN,0.000000,0.000000,0.000000,0.000000,NaN
25%,15651.000000,NaN,1.355645e+02,1.000000,1.278594e+02,123.181045,402.382410,102.964970,1.257239e+02,122.838310,111.320385,119.408475,124.694930,125.401905,125.560290,1.256872e+02,0.000000,0.000000,0.223550,-2.101307,-0.000000,0.000000,1.000000,1.0,0.712137,1.000000,1.0,1.635296,1.632366,1.696011,0.000000,0.000000,0.000000e+00,0.000000,0.876325,4.141230e-18,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.0,NaN,2.924813e-01,0.930345,NaN,0.000000,0.000000,0.000000,0.000000,NaN
50%,57011.000000,NaN,2.688260e+02,1.000000,2.283896e+02,205.899780,3544.591710,143.852060,2.169817e+02,201.525800

# 1 - Removing Features

In this section some features are removed. The reasons to remove a particular feature are:
- That feature is not available in real-case scenario: it is only available in a simulation context
- That feature is *redundant* with some other features

## Remove `event_index` feature

In [6]:
if "event_index" in ds.columns.to_list():
    ds = ds.drop(columns=["event_index"])
    print(f"Column 'event_index' removed from dataset, shape is now {ds.shape}.")
else:
    print(f"Column 'event_index' not found in dataset, shape is {ds.shape}.")

Column 'event_index' removed from dataset, shape is now (368775, 53).


## Remove `primary_e0_kev` feature

The `primary_e0_kev` feature is **not available** in real-case scanario: it is the energy of the event at the start of the simulation (before hitting the detector).

In [7]:
if "primary_e0_kev" in ds.columns.to_list():
    ds = ds.drop(columns=["primary_e0_kev"])
    print(f"Column 'primary_e0_kev' removed, shape is now {ds.shape}.")
else:
    print(f"Column 'primary_e0_kev' not found, shape is {ds.shape}.")

Column 'primary_e0_kev' removed, shape is now (368775, 52).


## Remove *total hit*-like feature

The features `n_active`, `hits_sum`, `n_voxels` are redundant: all these features represents the number of activated sub-cubes (detector). Follow a demonstration:

In [8]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]):
    ds["hits_sum"] = ds["hits_sum"].astype(int)
    #print(f"Does the columns ['n_voxels', 'hits_sum','n_active'] have the same values? {ds["n_active"].equals(ds["n_voxels"]) and ds["n_active"].equals(ds["hits_sum"])}")
    print(f"n_voxels = hits_sum : {ds['n_voxels'].equals(ds['hits_sum'])}")
    print(f"n_voxels = n_active : {ds['n_voxels'].equals(ds['n_active'])}")
    print(f"hits_sum = n_active : {ds['hits_sum'].equals(ds['n_active'])}")

n_voxels = hits_sum : True
n_voxels = n_active : True
hits_sum = n_active : True


Totally drop `n_active` and `n_voxels`:

In [9]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]): 
    ds = ds.drop(columns=["n_voxels", "n_active"])
    print(f"Columns 'n_voxels' and 'n_active' removed, shape is now {ds.shape}.")
else: 
    print(f"Columns 'n_voxels' and 'n_active' not found, shape is {ds.shape}.")  

Columns 'n_voxels' and 'n_active' removed, shape is now (368775, 50).


## Remove single-valued feature

The `hits_mean_active`, `eff_E_ge_T` and `frac_active` feature contain just one unique value across avery sample:

In [10]:
if "hits_mean_active" in ds.columns.to_list():
    print(ds["hits_mean_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['hits_mean_active'].nunique()}) in 'hits_mean_active': {ds['hits_mean_active'].value_counts().max()} / {ds.shape[0]}")

count    368775.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: hits_mean_active, dtype: float64

Number of same value (1) in 'hits_mean_active': 368775 / 368775


In [11]:
if "frac_active" in ds.columns.to_list():
    print(ds["frac_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['frac_active'].nunique()}) in 'frac_active': {ds['frac_active'].value_counts().max()} / {ds.shape[0]}")

count    368775.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: frac_active, dtype: float64

Number of same value (1) in 'frac_active': 368775 / 368775


In [12]:
if "eff_E_ge_T" in ds.columns.to_list():
    print(ds["eff_E_ge_T"].describe(include="all"))
    print(f"\nNumber of same value ({ds['eff_E_ge_T'].nunique()}) in 'eff_E_ge_T': {ds['eff_E_ge_T'].value_counts().max()} / {ds.shape[0]}")

count    368775.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: eff_E_ge_T, dtype: float64

Number of same value (1) in 'eff_E_ge_T': 368775 / 368775


Totally drop the columns:

In [13]:
if "hits_mean_active" in ds.columns.to_list():
    ds = ds.drop(columns=["hits_mean_active"])
    print(f"Column 'hits_mean_active' removed, shape is now {ds.shape}.")
if "frac_active" in ds.columns.to_list():
    ds = ds.drop(columns=["frac_active"])
    print(f"Column 'frac_active' removed, shape is now {ds.shape}.")
if "eff_E_ge_T" in ds.columns.to_list():
    ds = ds.drop(columns=["eff_E_ge_T"])
    print(f"Column 'eff_E_ge_T' removed, shape is now {ds.shape}.")

Column 'hits_mean_active' removed, shape is now (368775, 49).
Column 'frac_active' removed, shape is now (368775, 48).
Column 'eff_E_ge_T' removed, shape is now (368775, 47).


# 2 - Handling NaN features

There are feature that contains only NaN values: 

In [14]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

nan_perc_df = pd.DataFrame(
    {
        "Column": nan_counts.index, 
        "Total NaN": nan_counts.values, 
        "Percentage": (nan_counts.values / ds.shape[0] * 100).round(2)
    }
).sort_values(by="Percentage", ascending=False).reset_index(drop=True)

nan_perc_df

,Column,Total NaN,Percentage
0,corr_E_hits,368775,100.00
1,f_occ,368775,100.00
2,D_f_r2,290698,78.83
3,E_skew,284028,77.02
4,E_kurt,284028,77.02
5,D_f,271294,73.57
6,E_std,242025,65.63
7,linearity,242025,65.63
8,planarity,242025,65.63
9,sphericity,242025,65.63


Drop those columns with 100% of NaN values:

In [15]:
columns_100perc_nan = nan_perc_df.iloc[:2]["Column"].tolist()  
ds = ds.drop(columns=columns_100perc_nan)
print(f"Columns {columns_100perc_nan} removed, shape is now {ds.shape}.")


Columns ['corr_E_hits', 'f_occ'] removed, shape is now (368775, 45).


## Hits/Shape and NaN correlation

In [16]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

onehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 1)).sum())
twohit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 2)).sum())
treehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 3)).sum())
morethantreehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] > 3)).sum())

pd.DataFrame(
    {
        "NaN Features": nan_counts.index,
        "Percentage of NaN values": (nan_counts.values / ds.shape[0] * 100).round(2),
        "NaN with 1 hit": onehit_nan_counts.values,
        "NaN with 2 hits": twohit_nan_counts.values,
        "NaN with 3 hits": treehit_nan_counts.values,
        "NaN with >3 hits": morethantreehit_nan_counts.values 
    }
).sort_values(by=["NaN with 1 hit", "NaN with 2 hits", "NaN with 3 hits", "NaN with >3 hits"], ascending=False).reset_index(drop=True)

,NaN Features,Percentage of NaN values,NaN with 1 hit,NaN with 2 hits,NaN with 3 hits,NaN with >3 hits
0,D_f_r2,78.83,242025,42003,5728,942
1,E_skew,77.02,242025,42003,0,0
2,E_kurt,77.02,242025,42003,0,0
3,D_f,73.57,242025,25231,3474,564
4,E_std,65.63,242025,0,0,0
5,linearity,65.63,242025,0,0,0
6,planarity,65.63,242025,0,0,0
7,sphericity,65.63,242025,0,0,0


Most of NaN are due to the low quantity of hits per event. Let's visualize the correlation between the NaN values and the number of hits (=1, =2 or > 2)

In [17]:
one_hit = ds[ds["hits_sum"] == 1]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in bg]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit[one_hit['particle_type'] == b].shape[0]} / {ds[ds['particle_type'] == b].shape[0]}" for b in bg],
        "One Hit percentage": np.round([one_hit[one_hit['particle_type'] == b].shape[0]/ds[ds['particle_type'] == b].shape[0]*100 for b in bg], 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        #"One Hit count": f"{one_hit_counts} / {ds[ds["particle_type"] == bg].shape[0]}",
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
    }
)

Number of events with 1 hit: 242025 (65.63%)
Number of events with 2 or less hit: 284028 (77.02%)
Number of events with multiple hits: 84747 (22.98%)


,Background,One Hit count,One Hit percentage,Two or less Hits count,Two or less Hits percentage
0,AtmosphericNeutrons,6239 / 9962,62.63,7844 / 9962,78.74
1,SecondaryProtonsUpward,5902 / 11901,49.59,7249 / 11901,60.91
2,PrimaryAlphas,2786 / 11677,23.86,4047 / 11677,34.66
3,CosmicPhotons,137476 / 152860,89.94,150935 / 152860,98.74
4,PrimaryProtons,3575 / 18461,19.37,5710 / 18461,30.93
5,AlbedoPhotons,65667 / 86862,75.60,78072 / 86862,89.88
6,PrimaryElectrons,1337 / 8537,15.66,2017 / 8537,23.63
7,PrimaryPositrons,3396 / 19636,17.29,4942 / 19636,25.17
8,SecondaryElectrons,3134 / 8730,35.90,4523 / 8730,51.81
9,SecondaryProtonsDownward,5486 / 11060,49.60,6746 / 11060,60.99


⚠️ Datasets must be balanced!

In [18]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

columns_with_nan = nan_counts.index

less2_hit_nan_counts = less2_hit[columns_with_nan].isna().sum()
multi_hit_nan_counts = multi_hit[columns_with_nan].isna().sum()
less2_hit_nan_counts_percent = np.round(less2_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)
multi_hit_nan_counts_percent = np.round(multi_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)

nan_report = pd.DataFrame({
    "NaN Count": nan_counts,
    "Two or less Hits NaN Count": less2_hit_nan_counts,
    "Multi Hits NaN Count": multi_hit_nan_counts,
    "Two or less Hits NaN percentage": less2_hit_nan_counts_percent,
    "Multi Hits NaN percentage": multi_hit_nan_counts_percent
})

nan_report.index.name = "Column"
nan_report.reset_index()

,Column,NaN Count,Two or less Hits NaN Count,Multi Hits NaN Count,Two or less Hits NaN percentage,Multi Hits NaN percentage
0,E_std,242025,242025,0,100.00,0.00
1,E_skew,284028,284028,0,100.00,0.00
2,E_kurt,284028,284028,0,100.00,0.00
3,linearity,242025,242025,0,100.00,0.00
4,planarity,242025,242025,0,100.00,0.00
5,sphericity,242025,242025,0,100.00,0.00
6,D_f,271294,267256,4038,98.51,1.49
7,D_f_r2,290698,284028,6670,97.71,2.29


The geometrical features and some energy features are obviously correlated with the number of hits: to define some of these features, at least 2 hits are required (e.g., standard deviation). In particular:
- E_std needs at least 2 hits,
- linearity, planarity and sphericity need at least 2,3+ hits
- E_skew (skewness) at lest 3 hits
- E_kurt 

Two different approach are possible: 
- setting those feature to a default value (e.g., 0) if the number of hits is equal to 1 (or <=2 or = 2)
- create differents sub-datasets:
    - One-hit dataset which does not contain the NaN features
    - 2-hit dataset 
    - *"Less than 2"*-hit dataset
    - Multi-hit dataset and handle separately the remaining NaNs 

# 3 - Distinguish events by number of hit

In [19]:
one_hit = ds[ds["hits_sum"] == 1]
two_hit = ds[ds["hits_sum"] == 2]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 hits: {two_hit.shape[0]} ({two_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
one_hit_counts = [one_hit[one_hit["particle_type"] == b].shape[0] for b in bg]
one_hit_percentages = [one_hit[one_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
two_hit_counts = [two_hit[two_hit["particle_type"] == b].shape[0] for b in bg]
two_hit_percentages = [two_hit[two_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in bg]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
multi_hit_counts = [multi_hit[multi_hit["particle_type"] == b].shape[0] for b in bg]
multi_hit_percentages = [multi_hit[multi_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "One Hit percentage": np.round(one_hit_percentages, 2),
        "Two Hits count": [f"{two_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "Two Hits percentage": np.round(two_hit_percentages, 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
        "Multiple Hits count": [f"{multi_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "Multiple Hits percentage": np.round(multi_hit_percentages, 2)
    }
).sort_values(by="Two or less Hits percentage", ascending=False)

Number of events with 1 hit: 242025 (65.63%)
Number of events with 2 hits: 42003 (11.39%)
Number of events with 2 or less hit: 284028 (77.02%)
Number of events with multiple hits: 84747 (22.98%)


,Background,One Hit count,One Hit percentage,Two Hits count,Two Hits percentage,Two or less Hits count,Two or less Hits percentage,Multiple Hits count,Multiple Hits percentage
3,CosmicPhotons,137476 / 152860,89.94,13459 / 152860,8.80,150935 / 152860,98.74,1925 / 152860,1.26
5,AlbedoPhotons,65667 / 86862,75.60,12405 / 86862,14.28,78072 / 86862,89.88,8790 / 86862,10.12
0,AtmosphericNeutrons,6239 / 9962,62.63,1605 / 9962,16.11,7844 / 9962,78.74,2118 / 9962,21.26
9,SecondaryProtonsDownward,5486 / 11060,49.60,1260 / 11060,11.39,6746 / 11060,60.99,4314 / 11060,39.01
1,SecondaryProtonsUpward,5902 / 11901,49.59,1347 / 11901,11.32,7249 / 11901,60.91,4652 / 11901,39.09
8,SecondaryElectrons,3134 / 8730,35.90,1389 / 8730,15.91,4523 / 8730,51.81,4207 / 8730,48.19
10,SecondaryPositrons,7027 / 29089,24.16,4916 / 29089,16.90,11943 / 29089,41.06,17146 / 29089,58.94
2,PrimaryAlphas,2786 / 11677,23.86,1261 / 11677,10.80,4047 / 11677,34.66,7630 / 11677,65.34
4,PrimaryProtons,3575 / 18461,19.37,2135 / 18461,11.56,5710 / 18461,30.93,12751 / 18461,69.07
7,PrimaryPositrons,3396 / 19636,17.29,1546 / 19636,7.87,4942 / 19636,25.17,14694 / 19636,74.83


Two possible approach:
- mantain one single dataset with a *flag*-like feature which distinguish between single or multi hits 
- divide the dataset in sub-datasets

NaN values must be handled!

In [20]:
one_hit = ds[ds["hits_sum"] == 1]
two_hit = ds[ds["hits_sum"] == 2]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

### One-hit Dataset (and *"2 or less"*-hit)

In [21]:
print(f"\"Raw\" One hit dataset shape: {one_hit.shape}")
print(f"\n\nNaN values in one hit dataset:")
oh_nan_val = one_hit.isna().sum()
oh_nan_val = oh_nan_val[oh_nan_val > 0]
print(oh_nan_val)

"Raw" One hit dataset shape: (242025, 45)


NaN values in one hit dataset:
E_std         242025
E_skew        242025
E_kurt        242025
linearity     242025
planarity     242025
sphericity    242025
D_f           242025
D_f_r2        242025
dtype: int64


In [22]:
print(f"\"Raw\" Less 2 hit dataset shape: {less2_hit.shape}")
print(f"\n\nNaN values in less 2 hit dataset:")
l2_nan_val = less2_hit.isna().sum()
l2_nan_val = l2_nan_val[l2_nan_val > 0]
print(l2_nan_val)

"Raw" Less 2 hit dataset shape: (284028, 45)


NaN values in less 2 hit dataset:
E_std         242025
E_skew        284028
E_kurt        284028
linearity     242025
planarity     242025
sphericity    242025
D_f           267256
D_f_r2        284028
dtype: int64


Just simply drop this features

In [23]:
## less one hit dataset cleaning
columns_to_drop = oh_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in one hit dataset: {columns_to_drop}")
    one_hit = one_hit.drop(columns=columns_to_drop)

    print(f"NaN values in one hit dataset after dropping NaN values: {one_hit.isna().sum().sum()}")

print(f'"Cleaned" One hit dataset shape: {one_hit.shape}')

## less 2 hit dataset cleaning
columns_to_drop = l2_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in less 2 hit dataset: {columns_to_drop}")
    less2_hit = less2_hit.drop(columns=columns_to_drop)

    print(f"NaN values in less 2 hit dataset after dropping NaN values: {less2_hit.isna().sum().sum()}")

print(f'"Cleaned" Less 2 hit dataset shape: {less2_hit.shape}')


Dropping columns with NaN values in one hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in one hit dataset after dropping NaN values: 0
"Cleaned" One hit dataset shape: (242025, 37)

Dropping columns with NaN values in less 2 hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in less 2 hit dataset after dropping NaN values: 0
"Cleaned" Less 2 hit dataset shape: (284028, 37)


In [24]:
one_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,242025,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.000000,242025.0,242025.0,242025.0,2.420250e+05,242025.0,242025.0,242025.000000,242025.000000,242025.000000,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025.0,242025
unique,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,1827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,137476
mean,NaN,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,1161.752115,0.0,0.0,0.0,3.287572e-17,1.0,1.0,4.485871,4.478269,4.523702,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
std,NaN,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,4615.051920,0.0,0.0,0.0,7.886120e-17,0.0,0.0,3.309909,3.247965,3.286904,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
min,NaN,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,0.0,0.0,-0.0,0.000000e+00,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
25%,NaN,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,115.031010,0.0,0.0,0.0,0.000000e+00,1.0,1.0,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
50%,NaN,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,151.190410,0.0,0.0,0.0,0.000000e+00,1.0,1.0,4.000000,5.000000,5.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
75%,NaN,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,258.369340,0.0,0.0,0.0,0.000000e+00,1.0,1.0,8.000000,8.000000,8.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN


In [25]:
less2_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,284028,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,2.840280e+05,284028.000000,284028.0,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,2.840280e+05,2.840280e+05,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.000000,284028.0,284028.000000,284028
unique,16635,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,1827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,150935
mean,NaN,1823.400720,1406.674545,1195.037251,1618.311840,1406.674545,1237.364710,1300.855898,1512.493193,1575.984381,1597.148110,1614.079094,211.637295,211.637295,0.076114,1.782080e-02,0.961700,1.0,4.482004,4.475644,4.509373,0.122146,0.158711,4.226922e-18,1.016310e-19,-0.000954,0.000150,0.002588,0.138782,0.139701,0.143408,1.147883,0.088833,0.105890,0.0,0.308901,NaN
std,NaN,6650.576841,4938.045164,4546.240459,5678.810703,4938.045164,4591.253729,4691.238591,5272.382181,5508.546247,5592.472022,5661.355189,1440.245333,1440.245333,0.195179,5.051401e-02,0.106486,0.0,3.276005,3.215262,3.253740,0.379199,0.877827,5.473441e-17,3.766443e-18,0.165789,0.167980,0.168670,0.550881,0.547349,0.564414,0.354985,0.284503,0.347451,0.0,0.941035,NaN
min,NaN,15.001710,15.001710,15.001220,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,15.001710,0.000000,0.000000,-0.000000,0.000000e+00,0.500007,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,-0.999281,-0.999281,-0.999486,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
25%,NaN,118.147802,114.645775,111.453680,116.317930,114.645775,112.456430,113.269072,115.650210,116.074255,116.206182,116.301457,0.000000,0.000000,-0.000000,0.000000e+00,1.000000,1.0,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
50%,NaN,164.145650,156.554495,150.670410,160.732360,156.554495,152.986340,154.345455,158.875970,160.022310,160.376805,160.676600,0.000000,0.000000,-0.000000,0.000000e+00,1.000000,1.0,4.362083,4.846060,5.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
75%,NaN,356.408770,310.977490,276.940375,329.646938,310.977490,293.742320,301.986478,320.537325,325.896240,327.811058,329.238702,0.000000,0.000000,0.000000,2.220446e-16,1.000000,1.0,8.000000,8.000000,8.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN


Almost every faeture is meaningless if the number of hits is equal to 1 (or <=2). Drop those feature:

In [26]:
one_hit = one_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]
less2_hit = less2_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]

### Two-hit Dataset

In [27]:
print(f"\"Raw\" Two hit dataset shape: {two_hit.shape}")
print(f"\n\nNaN values in two hit dataset:")
th_nan_val = two_hit.isna().sum()
th_nan_val = th_nan_val[th_nan_val > 0]
print(th_nan_val)

"Raw" Two hit dataset shape: (42003, 45)


NaN values in two hit dataset:
E_skew    42003
E_kurt    42003
D_f       25231
D_f_r2    42003
dtype: int64


Just simply drop full nan features and impute 0 to `D_f`:

In [28]:
columns_to_drop = th_nan_val[th_nan_val == two_hit.shape[0]].index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with all NaN values in two hit dataset: {columns_to_drop}")
    two_hit = two_hit.drop(columns=columns_to_drop)

    print(f"NaN values in two hit dataset after dropping all NaN values: {two_hit.isna().sum().sum()}")

## impute 0 to D_f 
if "D_f" in two_hit.columns:
    two_hit["D_f"] = two_hit["D_f"].fillna(0)
    print(f"NaN values in two hit dataset after imputing 0 to D_f: {two_hit.isna().sum().sum()}")


Dropping columns with all NaN values in two hit dataset: ['E_skew', 'E_kurt', 'D_f_r2']
NaN values in two hit dataset after dropping all NaN values: 25231
NaN values in two hit dataset after imputing 0 to D_f: 0


In [29]:
two_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_std,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,linearity,planarity,sphericity,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,D_f,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,42003,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.0,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,4.200300e+04,4.200300e+04,4.200300e+04,4.200300e+04,4.200300e+04,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.000000,42003.0,4.200300e+04,42003.000000,42003.000000,42003.0,42003.000000,42003
unique,16135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,"[0,9,0];[0,8,0]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13459
mean,NaN,5635.878489,2817.939244,2023.895312,1386.829145,4249.049344,2817.939244,1673.051165,2102.384195,3533.494294,3962.827324,4105.938334,4220.427142,1431.110099,1431.110099,0.514692,0.120506,0.741012,1.0,4.459721,4.460518,4.426810,0.825962,1.073216,2.858282e-17,6.872380e-19,1.000000e+00,1.345188e-17,3.021902e-19,-0.006449,0.001014,0.017500,0.938457,0.944671,0.969740,2.0,6.489036e-17,0.600695,0.716039,0.0,2.088819,NaN
std,NaN,12621.769237,6310.884618,4956.145266,4122.271511,9339.420904,6310.884618,4426.610531,5033.862488,7775.320042,8705.182888,9021.068623,9275.565286,3504.523926,3504.523926,0.178517,0.069861,0.139721,0.0,3.073308,3.019927,3.054366,0.625310,2.056537,1.398659e-16,9.773795e-18,2.814862e-17,3.081830e-17,2.799016e-18,0.431080,0.436819,0.438317,1.140899,1.124922,1.163122,0.0,8.549826e-17,0.489761,0.615993,0.0,1.506735,NaN
min,NaN,30.456390,15.228195,0.003062,15.001220,15.448160,15.228195,15.052223,15.118212,15.338178,15.404167,15.426163,15.443761,0.002165,0.002165,0.002380,0.000003,0.500007,1.0,0.000000,0.000000,0.000000,0.023788,0.000566,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,-0.999281,-0.999281,-0.999486,0.000000,0.000000,0.000000,2.0,-9.309075e-17,0.000000,0.000000,0.0,1.000000,NaN
25%,NaN,212.387205,106.193602,49.592100,43.060300,154.138545,106.193602,56.588940,76.795255,131.823184,145.554051,150.003687,153.244156,35.066910,35.066910,0.405599,0.060429,0.620859,1.0,1.621599,1.627241,1.587153,0.450171,0.202654,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,-0.177132,-0.178586,-0.150256,0.000000,0.000000,0.000000,2.0,0.000000e+00,0.000000,0.000000,0.0,1.000000,NaN
50%,NaN,578.000990,289.000495,170.379818,140.420090,405.524030,289.000495,192.565682,230.914900,345.065790,382.179560,393.890693,403.140324,120.476725,120.476725,0.579415,0.116914,0.733827,1.0,4.388316,4.486093,4.406432,0.626065,0.391957,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,2.0,0.000000e+00,1.000000,1.000000,0.0,1.414214,NaN
75%,NaN,5014.313465,2507.156733,1669.212192,510.998910,4069.535755,2507.156733,989.338164,1623.070248,3317.580397,3776.586283,3921.645190,4037.257398,1180.311260,1180.311260,0.663642,0.179825,0.859650,1.0,7.342751,7.210069,7.207370,1.039065,1.079655,6.281471e-18,0.000000e+00,1.000000e+00,7.601352e-18,0.000000e+00,0.159005,0.181286,0.224758,1.000000,1.000000,1.000000,2.0,1.451502e-16,1.000000,1.000000,0.0,2.449490,NaN


Almost every faeture is meaningless if the number of hits is equal to 2. Drop those feature:

In [30]:
two_hit = two_hit[["pixel_ijk", "E_sum", "E_min", "E_max", "com_x", "com_y", "com_z", "rg", "span_x", "span_y", "span_z", "D_f", "hits_sum", "particle_type"]]

### Multi-hit Dataset

In [31]:
print(f"\"Raw\" Multi hits dataset shape: {multi_hit.shape}")
print(f"\n\nNaN values in multi hits dataset:")
mh_nan_val = multi_hit.isna().sum()
mh_nan_val = mh_nan_val[mh_nan_val > 0]
print(mh_nan_val)

"Raw" Multi hits dataset shape: (84747, 45)


NaN values in multi hits dataset:
D_f       4038
D_f_r2    6670
dtype: int64


<!-- To compute the skewness and the kurtosis coeff. at least 3 hits are needed. To avoid to create a third dataset, and since there are almost ~40% missing values, we can decide to totally drop these features. -->

R2 index may be dropped. For the Fractal Dimension we decide to impute a default value 0, preserving the meaning of the feature: a too small to be computed fractal dimension may be set to 0.  


In [32]:
# mh_cols_drop = ["E_skew", "E_kurt", "D_f_r2"]
mh_cols_drop = ["D_f_r2"]

if all(col in multi_hit.columns.to_list() for col in mh_cols_drop):
    print(f"\nDropping columns with NaN values in multi hits dataset: {mh_cols_drop}")
    multi_hit = multi_hit.drop(columns=mh_cols_drop)

    print(f"NaN values in multi hits dataset after dropping NaN values: {multi_hit.isna().sum().sum()}")

# impute 0 to D_f
if "D_f" in multi_hit.columns.to_list():
    multi_hit["D_f"] = multi_hit["D_f"].fillna(0)
    print(f"NaN values in 'D_f' column after imputation: {multi_hit['D_f'].isna().sum()}")

print(f'"Cleaned" Multi hits dataset shape: {multi_hit.shape}')


Dropping columns with NaN values in multi hits dataset: ['D_f_r2']
NaN values in multi hits dataset after dropping NaN values: 4038
NaN values in 'D_f' column after imputation: 0
"Cleaned" Multi hits dataset shape: (84747, 44)


### Summary and saving 

In [33]:
print(f"Final shape of one hit dataset: {one_hit.shape}")
print(f"Final shape of two hit dataset: {two_hit.shape}")
print(f"Final shape of less 2 hit dataset: {less2_hit.shape}")
print(f"Final shape of multi hits dataset: {multi_hit.shape}")

Final shape of one hit dataset: (242025, 7)
Final shape of two hit dataset: (42003, 14)
Final shape of less 2 hit dataset: (284028, 7)
Final shape of multi hits dataset: (84747, 44)


In [34]:
if "multi_hit" in one_hit.columns.to_list():
    one_hit = one_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from one_hit dataset, shape is now {one_hit.shape}.")
if "multi_hit" in two_hit.columns.to_list():
    two_hit = two_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from two_hit dataset, shape is now {two_hit.shape}.")
if "multi_hit" in less2_hit.columns.to_list():
    less2_hit = less2_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from less2_hit dataset, shape is now {less2_hit.shape}.")
if "multi_hit" in multi_hit.columns.to_list():
    multi_hit = multi_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from multi_hit dataset, shape is now {multi_hit.shape}.")

In [35]:
if SAVE:
    one_hit.to_csv(file_path + "/one_hit_dataset.csv", index=False)
    two_hit.to_csv(file_path + "/two_hit_dataset.csv", index=False)
    less2_hit.to_csv(file_path + "/less2_hit_dataset.csv", index=False)
    multi_hit.to_csv(file_path + "/multi3plus_hit_dataset.csv", index=False)